# 4. 读出重复性诊断：旧开发集、固定预测器

默认 Gemma-12B。

[实验设置与运行说明](../../../docs/experiment_notes/03_section44_readout_correction/6fb890c32c39ac57.md)


In [1]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 17 tracked cells.')

IMPLEMENTATION_SHA256 = '160372af558929c42134517d3ffc09527278d018135dd39a46f32eb6987dde7d'
IMPLEMENTATION_SHA256 = 'f57efd1987491533d8e8c0c54f8aa29db42e4f6e1fcfb49d469780fd6c1b7573'


Initialize progress display:   0%|          | 0/1 [00:00<?, ?cell/s]

Cell progress helper ready: 17 tracked cells.


In [2]:
%%cell_progress 1/12 Mount Drive
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


Mount Drive:   0%|          | 0/12 [00:00<?, ?cell/s]

Mounted at /content/drive
Project: /content/drive/MyDrive/VisualTorque


In [3]:
%%cell_progress 2/12 Install inference environment
import importlib
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'hf_xet>=1.1', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'scikit-learn>=1.5',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


Install inference environment:   8%|8         | 1/12 [00:00<?, ?cell/s]

torch: 2.11.0+cu128 | CUDA: 12.8
torchvision: 0.26.0+cu128
Toolkit source: /content/drive/MyDrive/VisualTorque/toolkit/src/visual_torque_toolkit/__init__.py


In [4]:
%%cell_progress 3/12 Check GPU access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


Check GPU access:  17%|#6        | 2/12 [00:00<?, ?cell/s]

NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA memory (GB): 95.0
Hugging Face login skipped: Secret HF_TOKEN does not exist.


In [5]:
%%cell_progress 4/12 Load shared validation utilities
"""Pure CPU utilities embedded in the follow-up notebooks; no model execution."""
import hashlib
import json
from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations, product
import numpy as np
import pandas as pd


def plain(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=plain, allow_nan=False).encode()).hexdigest()


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def read_jsonl(path):
    return [json.loads(s) for s in Path(path).read_text(encoding='utf-8').splitlines() if s.strip()]


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=plain, allow_nan=False) + '\n', encoding='utf-8')
    temp.replace(path)


def freeze(path, identity):
    identity = json.loads(json.dumps(identity, default=plain, sort_keys=True, allow_nan=False))
    if Path(path).is_file():
        assert read_json(path) == identity, f'Configuration changed; preserve the old run and use a new RUN_ID: {path}'
    else:
        save_json(path, identity)
    return digest(identity)


def cached_json(directory, key, identity, compute):
    path = Path(directory) / (digest(str(key))[:24] + '.json')
    expected = digest(identity)
    if path.is_file():
        old = read_json(path)
        assert old['key'] == str(key) and old['identity'] == expected, f'Incompatible cache: {path}'
        return old['result']
    value = compute()
    save_json(path, dict(key=str(key), identity=expected, result=value))
    return value


def bootstrap(values, key='', replicates=10000):
    x = np.asarray(values, dtype=float)
    assert x.ndim == 1 and len(x) and np.isfinite(x).all()
    rng = np.random.default_rng(int(digest(str(key))[:16], 16))
    draws = np.concatenate([rng.choice(x, (min(250, replicates-i), len(x)), replace=True).mean(1)
                            for i in range(0, replicates, 250)])
    return dict(n=len(x), mean=float(x.mean()), lower=float(np.quantile(draws, .025)),
                upper=float(np.quantile(draws, .975)),
                p_value=float((1 + (draws <= 0).sum()) / (replicates + 1)))


def bh(values):
    p = np.asarray(values, dtype=float)
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
    order = np.argsort(p)
    q = np.empty_like(p)
    previous = 1.
    for rank in range(len(p)-1, -1, -1):
        i = order[rank]
        previous = min(previous, p[i]*len(p)/(rank+1))
        q[i] = previous
    return q


def paired(left, right, id_col, value):
    assert left[id_col].is_unique and right[id_col].is_unique
    assert set(left[id_col]) == set(right[id_col]), 'Different paired support'
    out = left[[id_col, value]].merge(right[[id_col, value]], on=id_col, validate='one_to_one', suffixes=('_l', '_r'))
    return out[value+'_l'].to_numpy(float) - out[value+'_r'].to_numpy(float)


def semantic_scores(record):
    if 'semantic_logprobs' in record:
        return {str(k): float(v) for k, v in record['semantic_logprobs'].items()}
    values = record['candidate_logprobs']
    if isinstance(values, dict):
        return {str(record['label_to_semantic'][k]): float(v) for k, v in values.items()}
    return dict(zip(map(str, record['candidate_semantics']), map(float, values)))


def correct_margin(record):
    values = semantic_scores(record)
    gold = str(record['gold_semantic'])
    assert gold in values and len(values) >= 2
    return values[gold] - max(v for k, v in values.items() if k != gold)


def random_unit(shape, key):
    r = np.random.default_rng(int(digest(key)[:16], 16)).normal(size=shape)
    return r / np.linalg.norm(r)


def coordinate_clamp(pre, untreated, axis, cast):
    h, base, u = [np.asarray(x, dtype=np.float64) for x in (pre, untreated, axis)]
    u = u / np.linalg.norm(u)
    a = float(np.sum((h-base)*u))
    return np.asarray(cast(h-a*u), dtype=np.float64), a


def native_norm_edit(pre, direction, norm, cast, tolerance=.02, steps=48):
    h = np.asarray(pre, dtype=np.float64)
    r = np.asarray(direction, dtype=np.float64)
    assert np.isfinite(h).all() and np.isfinite(r).all() and norm >= 0 and np.linalg.norm(r) > 0
    r /= np.linalg.norm(r)
    if norm == 0:
        return h.copy(), dict(actual_norm=0., target_norm=0., relative_error=0., matched=True)
    lo, hi = 0., max(float(norm), 1e-12)
    for _ in range(32):
        if np.linalg.norm(np.asarray(cast(h+hi*r))-h) >= norm:
            break
        hi *= 2
    best, error = h.copy(), 1.
    for _ in range(steps):
        mid = (lo+hi)/2
        candidate = np.asarray(cast(h+mid*r), dtype=np.float64)
        actual = float(np.linalg.norm(candidate-h))
        current = abs(actual-norm)/norm
        if current < error:
            best, error = candidate.copy(), current
        if error <= tolerance:
            break
        if actual < norm:
            lo = mid
        else:
            hi = mid
    return best, dict(actual_norm=float(np.linalg.norm(best-h)), target_norm=float(norm),
                      relative_error=float(error), matched=bool(error <= tolerance))


def equal_removal(pre, untreated, axis, random_direction, cast, tolerance=.02):
    target, coefficient = coordinate_clamp(pre, untreated, axis, cast)
    target_norm = float(np.linalg.norm(target-np.asarray(pre)))
    sign = 1. if coefficient >= 0 else -1.
    post, audit = native_norm_edit(pre, -sign*np.asarray(random_direction), target_norm, cast, tolerance)
    audit.update(reference_coefficient=coefficient, reference_actual_norm=target_norm)
    return post, audit


def geometry_signature(stacks):
    a = tuple(sorted((str(s['side']), int(s['distance']), int(s['units'])) for s in stacks))
    b = tuple(sorted(('right' if side == 'left' else 'left', d, u) for side, d, u in a))
    return min(a, b)  # mirrors belong to the same physical family


def geometries_in(value):
    found = set()
    if isinstance(value, dict):
        stacks = value.get('stacks')
        if isinstance(stacks, list) and stacks and all(isinstance(s, dict) and {'side','distance','units'} <= set(s) for s in stacks):
            found.add(geometry_signature(stacks))
        # Some prior matched cohorts store geometry instead of stack records.
        geom = value.get('pair_geometry')
        if isinstance(geom, (list, tuple)) and len(geom) == 4 and all(isinstance(x, (int,float)) for x in geom):
            u1,d1,u2,d2 = map(int, geom)
            found.add(geometry_signature([dict(side='left',distance=d1,units=u1),dict(side='right',distance=d2,units=u2)]))
        if isinstance(geom, (list, tuple)) and len(geom)==2 and all(isinstance(x,(list,tuple)) and len(x)==2 for x in geom):
            (u1,d1),(u2,d2)=geom
            found.add(geometry_signature([dict(side='left',distance=int(d1),units=int(u1)),dict(side='right',distance=int(d2),units=int(u2))]))
        for child in value.values():
            found.update(geometries_in(child))
    elif isinstance(value, list):
        for child in value:
            found.update(geometries_in(child))
    return found


def physics(stacks):
    units = {s:sum(int(x['units']) for x in stacks if x['side']==s) for s in ('left','right')}
    dist = {s:sum(int(x['distance']) for x in stacks if x['side']==s) for s in ('left','right')}
    torque = {s:sum(int(x['units'])*int(x['distance']) for x in stacks if x['side']==s) for s in ('left','right')}
    out = lambda x:'left' if x>0 else 'right' if x<0 else 'balanced'
    net=torque['left']-torque['right']; w=units['left']-units['right']; d=dist['left']-dist['right']
    return dict(net_torque=net,outcome=out(net),weight_score=w,distance_score=d,
                normalized_margin=abs(net)/max(1,sum(torque.values())),
                left_units=units['left'],right_units=units['right'],left_torque=torque['left'],right_torque=torque['right'],
                strategy_predictions=dict(exact_torque=out(net),weight_only=out(w),distance_only=out(d)))


def exclusion_snapshot(project, followup_root, dataset_dir=None):
    lock=Path(followup_root)/'prior_geometry_snapshot.json'
    if lock.is_file():
        old=read_json(lock)
        for path, sha in old['files'].items():
            assert file_hash(Path(path))==sha, f'Prior input changed: {path}'
        return {tuple(tuple(x) for x in sig) for sig in old['signatures']}, old
    paths={Path(dataset_dir or project/'data/dataset_v1')/'scenes.jsonl'}
    for pattern in ('*manifest*.jsonl','*support*.jsonl','*geometries*.jsonl'):
        paths.update(p for p in (project/'results').rglob(pattern) if Path(followup_root) not in p.parents)
    excluded=set(); hashes={}
    for path in sorted(paths):
        assert path.is_file(), f'Missing required geometry input: {path}'
        excluded.update(geometries_in(read_jsonl(path)))
        hashes[str(path.resolve())]=file_hash(path)
    assert excluded, 'No prior physical geometry was indexed'
    old=dict(files=hashes,signatures=sorted(excluded),mirror_canonical=True)
    save_json(lock,old)
    return excluded,old


def inverse_families(n, seed, excluded):
    assert n%4==0
    pool=defaultdict(list)
    for dl in combinations(range(1,7),2):
        for dr in combinations(range(1,7),2):
            for units in product(range(1,6), repeat=4):
                stacks=[dict(side=side,distance=d,units=u) for side,d,u in zip(('left','left','right','right'),dl+dr,units)]
                f=physics(stacks)
                pool[dl+dr].append((stacks,f))
    candidates=[]
    for distances,rows in pool.items():
        by_net=defaultdict(list)
        for stacks,f in rows:
            if f['outcome']==f['strategy_predictions']['weight_only'] and f['outcome']!='balanced':
                by_net[f['net_torque']].append((stacks,f))
        for stacks,f in rows:
            margin=f['normalized_margin']; difficulty='hard' if .08<=margin<.22 else 'moderate' if .22<=margin<.45 else None
            if (difficulty and f['outcome']!='balanced' and f['strategy_predictions']['weight_only'] not in ('balanced',f['outcome'])
                    and f['strategy_predictions']['distance_only']==f['outcome'] and abs(f['distance_score'])>=2):
                sources=by_net.get(f['net_torque'],[])
                if sources:candidates.append((digest([seed,stacks]),difficulty,stacks,f,sources))
    candidates.sort(key=lambda r:r[0]); used=set(excluded); counts=Counter(); result=[]
    for _,difficulty,stacks,f,sources in candidates:
        stratum=(difficulty,f['outcome'])
        if counts[stratum]>=n//4 or geometry_signature(stacks) in used:continue
        valid=[(s,g) for s,g in sources if geometry_signature(s) not in used|{geometry_signature(stacks)}]
        if not valid:continue
        valid.sort(key=lambda x:(sum(abs(a['units']-b['units']) for a,b in zip(stacks,x[0])),digest([seed,x[0]])))
        src,sf=valid[0];fid='fixed_'+digest([seed,stacks,src])[:20]
        roles={name:dict(scene_id=fid+'__'+name,image_filename=fid+'__'+name+'.png',stacks=s,features=g)
               for name,s,g in [('base',stacks,f),('aligned_source',src,sf)]}
        result.append(dict(family_id=fid,remapping_split='new_fixed_confirmation',roles=roles,margin_bin=difficulty))
        used.update([geometry_signature(stacks),geometry_signature(src)]);counts[stratum]+=1
        if len(result)==n:break
    assert len(result)==n and set(counts.values())=={n//4}, (len(result),counts)
    return sorted(result,key=lambda r:r['family_id'])


def repair_families(n, seed, excluded):
    assert n%4==0
    candidates=defaultdict(list)
    for ul,dl,ur,dr in product(range(1,9),range(1,7),range(1,9),range(1,7)):
        s=[dict(side='left',units=ul,distance=dl),dict(side='right',units=ur,distance=dr)]
        f=physics(s)
        if f['net_torque']==0:continue
        if ul==ur and dl!=dr:kind='equal_weight_distance'
        elif f['strategy_predictions']['weight_only'] not in ('balanced',f['outcome']):kind='weight_distance_conflict'
        elif f['strategy_predictions']['weight_only']==f['outcome'] and f['strategy_predictions']['distance_only']==f['outcome']:kind='aligned_cues'
        else:continue
        candidates[kind].append(s)
    for units in product(range(1,6),repeat=4):
        for distances in [((1,3),(2,5)),((2,5),(1,4)),((1,6),(3,5)),((2,4),(3,6))]:
            s=[dict(side=side,distance=d,units=u) for side,d,u in zip(('left','left','right','right'),distances[0]+distances[1],units)]
            if physics(s)['net_torque']!=0:candidates['multi_stack_sum'].append(s)
    used=set(excluded); result=[]
    for kind in ('equal_weight_distance','weight_distance_conflict','multi_stack_sum','aligned_cues'):
        rows=sorted(candidates[kind],key=lambda s:digest([seed,kind,s])); count=0
        for stacks in rows:
            sig=geometry_signature(stacks)
            if sig in used:continue
            fid='repair_'+digest([seed,kind,sig])[:20];scenes=[]
            for mirror in (False,True):
                current=[dict(s,side=('right' if s['side']=='left' else 'left')) if mirror else dict(s) for s in stacks]
                for style in ('standard','alternate'):
                    sid=f'{fid}__m{int(mirror)}__{style}'
                    scenes.append(dict(scene_id=sid,image_filename=sid+'.png',stacks=current,features=physics(current),style=style,mirror=mirror))
            result.append(dict(family_id=fid,stratum=kind,scenes=scenes));used.add(sig);count+=1
            if count==n//4:break
        assert count==n//4,(kind,count,'Insufficient unused geometries; do not silently change task ranges')
    return sorted(result,key=lambda r:r['family_id'])


"""CPU protocol helpers for the independent confirmation notebooks.

Embedded after followup_core.py. These functions do not execute a model.
"""


def frozen_arrays(path, arrays):
    arrays = {k: np.asarray(v, dtype=np.float32) for k, v in arrays.items()}
    assert arrays and all(np.isfinite(v).all() for v in arrays.values())
    path = Path(path)
    if path.is_file():
        with np.load(path, allow_pickle=False) as old:
            assert set(old.files) == set(arrays) and all(np.array_equal(old[k], v) for k, v in arrays.items()), path
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        temp = path.with_suffix('.npz.tmp')
        with temp.open('wb') as f:
            np.savez_compressed(f, **arrays)
        temp.replace(path)
    return file_hash(path)


def fresh_geometry_index(project, dataset_dir, current_support):
    """Read all known old cohorts, including already started v1 confirmations."""
    current_support = Path(current_support).resolve()
    paths = {Path(dataset_dir) / 'scenes.jsonl'}
    for pattern in ('*manifest*.jsonl', '*support*.jsonl', '*geometries*.jsonl', 'support.json'):
        paths.update(p for p in (Path(project) / 'results').rglob(pattern)
                     if current_support not in p.resolve().parents)
    excluded, files = set(), {}
    for path in sorted(paths):
        value = read_jsonl(path) if path.suffix == '.jsonl' else read_json(path)
        signatures = geometries_in(value)
        if signatures:
            excluded.update(signatures)
            files[str(path.resolve())] = file_hash(path)
    assert excluded and str((Path(dataset_dir) / 'scenes.jsonl').resolve()) in files
    return excluded, files


def independent_support(project, dataset_dir, directory, n, seed, protocol):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    excluded, current_files = fresh_geometry_index(project, dataset_dir, directory)
    snapshot = directory / 'exclusions.json'
    if snapshot.is_file():
        old = read_json(snapshot)
        for name, sha in old['files'].items():
            assert file_hash(name) == sha, f'Historical geometry input changed: {name}'
    else:
        save_json(snapshot, dict(files=current_files, signatures=sorted(excluded)))
    config = dict(n=n, seed=seed, generator='four_strata_mirror_render_1',
                  exclusions_sha256=file_hash(snapshot), protocol=digest(protocol))
    freeze(directory / 'protocol.json', config)
    path = directory / 'support.json'
    if path.is_file():
        saved = read_json(path)
        assert saved['config'] == config
        families = saved['families']
    else:
        families = repair_families(n, seed, excluded)
        save_json(path, dict(config=config, families=families))
    assert len(families) == n and len({f['family_id'] for f in families}) == n
    assert len(geometries_in(families)) == n
    assert not geometries_in(families) & excluded, (
        'This cohort overlaps a previous or newly registered experiment; changing RUN_ID does not establish independence.')
    return families, path


def recover_gemma_candidate(source, manifest_path, dataset_scenes, revision, progress=lambda rows: rows):
    """Reconstruct the one specified candidate from old TRAIN caches only."""
    from sklearn.preprocessing import StandardScaler
    from sklearn.cross_decomposition import PLSRegression
    import sklearn
    source = Path(source)
    selection_path = source / 'selected_route.json'
    identity_path = source / 'development_identity.json'
    selection, identity = read_json(selection_path), read_json(identity_path)
    assert selection['model'] == 'gemma12b' and selection['revision'] == revision
    assert selection['eligible'] is False, 'This entry preserves the supplied negative development result.'
    assert selection['training_identity'] == digest(identity)
    assert identity['manifest'] == file_hash(manifest_path)
    manifest = read_jsonl(manifest_path)
    train_ids = sorted(str(r['scene_id']) for r in manifest if r['split'] == 'probe_train')
    dev_ids = sorted(str(r['scene_id']) for r in manifest if r['split'] == 'probe_dev')
    assert len(train_ids) == 235 and len(dev_ids) == 19 and not set(train_ids) & set(dev_ids)
    assert set(train_ids) == set(selection['fit_scenes'])
    scene_map = {str(r['scene_id']): r for r in read_jsonl(dataset_scenes)}
    states, targets, files = [], [], {str(selection_path): file_hash(selection_path),
        str(identity_path): file_hash(identity_path), str(manifest_path): file_hash(manifest_path),
        str(dataset_scenes): file_hash(dataset_scenes)}
    for sid in progress(train_ids):
        path = source / 'capture' / (digest(sid)[:24] + '.json')
        cached = read_json(path)
        assert cached['key'] == sid and cached['identity'] == digest(identity), path
        assert len(cached['result']['scores']) == 6
        assert len(identity['inputs'][sid]) == 6
        assert {e['split'] for e in identity['inputs'][sid]} == {'probe_train'}
        y_values = {int(e['torque']) for e in identity['inputs'][sid]}
        assert len(y_values) == 1
        target = next(iter(y_values))
        assert physics(scene_map[sid]['stacks'])['net_torque'] == target
        states.append(cached['result']['mean_hidden']['31'])
        targets.append(target)
        files[str(path)] = file_hash(path)
    X, y = np.asarray(states, float), np.asarray(targets, float)
    assert np.isfinite(X).all() and (y > 0).any() and (y < 0).any()
    scaler = StandardScaler().fit(X)
    Z = scaler.transform(X)
    pls = PLSRegression(n_components=4, scale=False, max_iter=1000).fit(Z, y.reshape(-1, 1))
    scores = pls.transform(Z)
    difference = scores[y > 0].mean(0) - scores[y < 0].mean(0)
    decoder = np.linalg.lstsq(scores - scores.mean(0), Z - Z.mean(0), rcond=1e-5)[0]
    direction = (difference @ decoder) * scaler.scale_ / 2
    assert np.isfinite(direction).all() and np.linalg.norm(direction) > 0
    dose = float(.01 * np.median(np.linalg.norm(X, axis=1)) / np.linalg.norm(direction))
    reference = dict(model='gemma12b', revision=revision, layer=31, factor=1., dose=dose,
        method='pls4_prototype_train_only_six_map_mean', source_eligible=False,
        source_selection_sha256=file_hash(selection_path), runtime=selection['runtime'],
        reconstruction=dict(numpy=np.__version__, sklearn=sklearn.__version__, files=files,
                            fit_scene_ids=train_ids, note='Original formula; no original selected vector was exported.'))
    return direction.astype('float32'), reference


def intervention_conditions(model, sign, upstream, semantic, code, untreated, key, seeds=(0, 1, 2, 3)):
    """Condition topology is fixed before observing the recipient's answer."""
    if model == 'gemma12b':
        add = {31: dict(action='add', payload=sign * upstream)}
        return [('zero', -1, {}), ('targeted', -1, add)] + [
            ('random', s, {31: dict(action='matched_add', reference=sign * upstream, key=key + ['random', s])})
            for s in seeds]
    assert model == 'qwen27b' and set(untreated) >= {49, 55}
    u = {48: dict(action='add', payload=sign * upstream)}
    a = {49: dict(action='add', payload=sign * semantic)}
    b = {55: dict(action='add', payload=code)}
    blocks = {49: dict(action='clamp', untreated=untreated[49], axis=sign * semantic),
              55: dict(action='clamp', untreated=untreated[55], axis=code)}
    rows = [('zero', -1, {}), ('u', -1, u), ('a', -1, a), ('b', -1, b),
            ('bypass', -1, {**u, 49: blocks[49], **b})]
    for name, layer, payload in [('u', 48, sign * upstream), ('a', 49, sign * semantic)]:
        for s in seeds:
            rows.append((name + '_random', s, {layer: dict(action='matched_add', reference=payload, key=key + [name, s])}))
    for layer in (49, 55):
        rows += [(f'self_{layer}', -1, {layer: blocks[layer]}),
                 (f'self_random_{layer}', 0, {layer: dict(blocks[layer], action='random_coordinate', key=key + ['self', layer, 0])})]
    for name, start, layer in [('ua', u, 49), ('ub', u, 55), ('ab', a, 55)]:
        rows.append((name + '_block', -1, {**start, layer: blocks[layer]}))
        rows.append((name + '_coordinate', 0, {**start, layer: dict(blocks[layer], action='random_coordinate', key=key + [name, 'coordinate', 0])}))
        for s in seeds:
            rows.append((name + '_equal', s, {**start, layer: dict(blocks[layer], action='equal_remove', key=key + [name, 'equal', s])}))
    assert len(rows) == 35 and len({(n, s) for n, s, _ in rows}) == len(rows)
    return rows


def reference_for_condition(model, condition):
    if model == 'gemma12b':
        return ('targeted', 31) if condition == 'random' else None
    return {'u_random': ('u', 48), 'a_random': ('a', 49),
            'ua_equal': ('ua_block', 49), 'ub_equal': ('ub_block', 55),
            'ab_equal': ('ab_block', 55)}.get(condition)


def audit_paired_writes(rows, model, tolerance=.02):
    """Also compare against the actually executed target condition, across forwards."""
    by_condition = {(r['condition'], r['random_seed']): r for r in rows}
    clean = by_condition[('zero', -1)]['record']
    for row in rows:
        reference = reference_for_condition(model, row['condition'])
        if reference:
            target, layer = reference
            target_norm = by_condition[(target, -1)]['record']['write_audits'][str(layer)]['actual_norm']
            actual = row['record']['write_audits'][str(layer)]['actual_norm']
            error = abs(actual - target_norm) / max(target_norm, 1e-12)
            row.update(paired_reference_condition=target, paired_reference_actual_norm=target_norm,
                       paired_actual_relative_error=error, valid=bool(row['valid'] and error <= tolerance))
        if row['condition'].startswith('self_'):
            scores = np.asarray(row['record']['candidate_logprobs'], float)
            zero = np.asarray(clean['candidate_logprobs'], float)
            gap = float(np.max(np.abs((scores - scores.mean()) - (zero - zero.mean()))))
            row.update(self_score_gap=gap, self_prediction_same=row['record']['predicted_semantic'] == clean['predicted_semantic'])
            row['valid'] = bool(row['valid'] and gap <= .15 and row['self_prediction_same'])
    return rows


def primary_contrasts(model):
    if model == 'gemma12b':
        return [('target_vs_zero', 'targeted', 'zero'), ('target_vs_random', 'targeted', 'random')]
    return [('u_vs_zero', 'u', 'zero'), ('u_vs_random', 'u', 'u_random'),
            ('a_vs_zero', 'a', 'zero'), ('a_vs_random', 'a', 'a_random'),
            ('ua_specific_loss', 'ua_equal', 'ua_block'),
            ('ub_specific_loss', 'ub_equal', 'ub_block'), ('ab_specific_loss', 'ab_equal', 'ab_block')]


def paired_summary(values, key, two_sided=False, replicates=10000):
    """Family percentile CI; paired sign-flip test (symmetric-null assumption)."""
    x = np.asarray(values, float)
    result = bootstrap(x, key, replicates)
    rng = np.random.default_rng(int(digest(['sign_flip', key])[:16], 16))
    observed = abs(x.mean()) if two_sided else x.mean()
    exceed = 0
    for offset in range(0, replicates, 250):
        signs = rng.choice([-1., 1.], size=(min(250, replicates - offset), len(x)))
        null = (signs * x).mean(1)
        if two_sided:
            null = abs(null)
        exceed += int((null >= observed - 1e-15).sum())
    result['p_value'] = (exceed + 1) / (replicates + 1)
    return result


def test_family_contrasts(frame, model, scope='all_attempted'):
    metrics = ['accuracy_change', 'margin_change']
    # Seeds first, then the fixed scene/mapping variants, then independent families.
    units = frame.groupby(['family_id', 'scene_id', 'variant', 'condition'])[metrics].mean().reset_index()
    means = units.groupby(['family_id', 'condition'])[metrics].mean().reset_index()
    tests = []
    for name, a, b in primary_contrasts(model):
        left, right = means[means.condition.eq(a)], means[means.condition.eq(b)]
        for metric in metrics:
            values = paired(left, right, 'family_id', metric)
            tests.append(dict(contrast=name, metric=metric, scope=scope,
                              **paired_summary(values, [model, scope, name, metric])))
    tests = pd.DataFrame(tests)
    tests['q_value'] = bh(tests.p_value)
    assert len(tests) == (4 if model == 'gemma12b' else 14)
    return tests, means


def confirmation_decisions(tests, frame, model):
    def passes(name, metric):
        row = tests[tests.contrast.eq(name) & tests.metric.eq(metric)].iloc[0]
        return bool(row.lower > 0 and row.q_value < .05)
    def valid(conditions):
        sub = frame[frame.condition.isin(conditions)]
        return bool(len(sub) and sub.valid.all())
    if model == 'gemma12b':
        result = {metric: bool(passes('target_vs_zero', metric) and passes('target_vs_random', metric)
                              and valid(['targeted', 'random'])) for metric in ('accuracy_change', 'margin_change')}
        return dict(accuracy_repair_confirmed=result['accuracy_change'], score_effect_confirmed=result['margin_change'],
                    next_step='eligible_for_separate_L31_to_L47_experiment' if result['accuracy_change'] else 'stop_this_candidate_no_rescan')
    result = {}
    for metric in ('accuracy_change', 'margin_change'):
        repair = {site: bool(passes(site + '_vs_zero', metric) and passes(site + '_vs_random', metric)
                            and valid([site, site + '_random'])) for site in ('u', 'a')}
        paths = {}
        for name, start, layer in [('ua', 'u', 49), ('ub', 'u', 55), ('ab', 'a', 55)]:
            paths[name] = bool(repair[start] and passes(name + '_specific_loss', metric)
                               and valid([name + '_equal', name + '_block', f'self_{layer}', f'self_random_{layer}']))
        result[metric] = dict(repair_confirmed=repair, selective_dependence_confirmed=paths)
    return result


def validate_trial_grid(frame, families, model):
    expected = [('zero', -1), ('targeted', -1)] + [('random', s) for s in range(4)] if model == 'gemma12b' else None
    if expected is None:
        mock = intervention_conditions('qwen27b', 1, np.ones(2), np.ones(2), np.ones(2), {49: np.zeros(2), 55: np.zeros(2)}, [])
        expected = [(n, s) for n, s, _ in mock]
    assert not frame.duplicated(['family_id', 'scene_id', 'variant', 'condition', 'random_seed']).any()
    assert set(frame.family_id) == {f['family_id'] for f in families}
    for family in families:
        sub = frame[frame.family_id.eq(family['family_id'])]
        assert set(sub.scene_id) == {s['scene_id'] for s in family['scenes']}
        assert len(sub) == 4 * 2 * len(expected)
        for _, unit in sub.groupby(['scene_id', 'variant']):
            assert set(zip(unit.condition, unit.random_seed)) == set(expected)
        assert set(sub.variant) == {'forward', 'reverse'}


def orthogonal_failure_table(records):
    frame = pd.DataFrame(records)
    sub = frame[frame.condition.eq('orthogonal')].copy()
    assert len(sub) and {'norm_matched', 'orthogonal_preserved', 'random_seed'} <= set(sub)
    assert sub.groupby('scene_id').random_seed.nunique().eq(4).all()
    assert sub.groupby('scene_id').size().eq(4).all()
    norm = sub.norm_matched.eq(True)
    preserve = sub.orthogonal_preserved.eq(True)
    known = sub.norm_matched.notna() & sub.orthogonal_preserved.notna()
    sub['failure_kind'] = np.select([~known, norm & preserve, ~norm & preserve, norm & ~preserve],
        ['missing_audit', 'valid', 'norm_only', 'readout_only'], default='both')
    sub['valid'] = known & norm & preserve
    return sub


Load shared validation utilities:  25%|##5       | 3/12 [00:00<?, ?cell/s]

In [6]:
%%cell_progress 5/12 Prepare existing Dataset v1
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


Prepare existing Dataset v1:  33%|###3      | 4/12 [00:00<?, ?cell/s]

Extracting Dataset v1:   0%|          | 0/2106 [00:00<?, ?file/s]

Dataset: /content/visualtorque_dataset_v1


In [7]:
%%cell_progress 6/12 Freeze diagnostic configuration

import math,hashlib,json
from datetime import datetime,timezone
RUN_ID='main'
MODEL_KEY='qwen9b'  # gemma12b / qwen9b；每个模型必须使用全新的运行时。
MODEL_SPECS={
 'qwen9b':dict(model_id='Qwen/Qwen3.5-9B',model_slug='Qwen__Qwen3.5-9B',expected_layers=32,
    revision='c202236235762e1c871ad0ccb60c8ee5ba337b9a',scan_layers=(31,),endpoint_layer=31,repair_layer=31,candidate_layers=(12,16,20,24,27,29)),
 'gemma12b':dict(model_id='google/gemma-4-12B-it',model_slug='google__gemma-4-12B-it',expected_layers=48,
    revision='707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7',scan_layers=(37,47),endpoint_layer=47,repair_layer=47,candidate_layers=(19,25,31,37,41,44)),
 'qwen27b':dict(model_id='Qwen/Qwen3.5-27B',model_slug='Qwen__Qwen3.5-27B__373612cb',expected_layers=64,
    revision='373612cb4f9288fcf550b526d86d85febedd9801',scan_layers=(49,63),endpoint_layer=63,repair_layer=55,candidate_layers=(25,33,41,49)),
}
assert MODEL_KEY in MODEL_SPECS
SPEC=MODEL_SPECS[MODEL_KEY];MODEL_ID=SPEC['model_id'];MODEL_REVISION=SPEC['revision'];MODEL_SLUG=SPEC['model_slug']
EXPECTED_LAYER_COUNT=SPEC['expected_layers'];MODEL_DTYPE='bfloat16';DEVICE_MAP='auto'
FOLLOWUP_ROOT=PROJECT_ROOT/'results/cross_model_followup_v1'
RANDOM_SEEDS=(0,1,2,3);SEED=20260913;PATCH_CONDITION_BATCH_SIZE=1
BOOTSTRAP_REPLICATES=10000;NORM_RELATIVE_TOLERANCE=.02
LABELS=('A','B','C');DONOR_TASKS=('weight_more','weight_less');TARGET_TASKS=('outcome',)
DIAGNOSTIC_VARIANTS=('forward','reverse');SITE_SCOPE='final_prompt_token'
def stable_key(value):return hashlib.sha256(f'{SEED}:{value!r}'.encode()).hexdigest()
def outcome(value):return 'left' if value>0 else 'right' if value<0 else 'balanced'
sha256_file=lambda p:'sha256:'+file_hash(p)
write_json_atomic=save_json

RUN_ID='main'
SOURCE_RUN_ID='main'  # 上一批实验的已完成目录；默认无需修改。
STAGE='readout_repeatability'
OUTPUT_ROOT=PROJECT_ROOT/'results/cross_model_independent_confirmation'/STAGE/RUN_ID/MODEL_SLUG
ANALYSIS_DIR=OUTPUT_ROOT/'analysis';ANALYSIS_DIR.mkdir(parents=True,exist_ok=True)

assert MODEL_KEY in ('gemma12b','qwen9b'), '本诊断只预定 Gemma 和 Qwen-9B。'
REPEATS=3
PROTOCOL=dict(schema='readout-repeatability-1.0',implementation=IMPLEMENTATION_SHA256,model=MODEL_KEY,
    revision=MODEL_REVISION,repeats=REPEATS,scenes=19,mappings=6,dtype=MODEL_DTYPE,batch_size=1,
    probe_refit=False,scene_scope='original probe_dev only',storage='native BF16 represented exactly as float32',
    prompt='original E1 dataset prompt; no repair-prompt substitution',new_scientific_cohort=False)
freeze(OUTPUT_ROOT/'protocol.json',PROTOCOL)


Freeze diagnostic configuration:  42%|####1     | 5/12 [00:00<?, ?cell/s]

'03d9113340ca04fa3592226d8833773a82f348cb01f238428effe61e6e4d9fc1'

'03d9113340ca04fa3592226d8833773a82f348cb01f238428effe61e6e4d9fc1'

In [8]:
%%cell_progress 7/12 Load the original frozen readout and nineteen development scenes
from visual_torque_toolkit.behavior import BehaviorRunConfig,PromptCondition,load_behavior_examples
READOUT_ROOT=PROJECT_ROOT/'results/cross_model_readout_correction_v2/models'/MODEL_SLUG
probe_path=READOUT_ROOT/'frozen_probe.json';old_protocol_path=READOUT_ROOT/'protocol.json'
old_runtime_path=READOUT_ROOT/'runtime_identity.json'
for p in (probe_path,old_protocol_path,old_runtime_path):assert p.is_file(),p
old_protocol=read_json(old_protocol_path);old_runtime=read_json(old_runtime_path)
assert old_protocol['model_id']==MODEL_ID and old_protocol['revision']==MODEL_REVISION
assert old_protocol['dtype']=='bfloat16' and old_protocol['batch_size']==1
probe=read_json(probe_path);SELECTED_LAYER=int(probe['layer']);CAPTURE_LAYERS=(SELECTED_LAYER,)
PROBE_MEAN=np.asarray(probe['mean'],dtype=np.float32);PROBE_SCALE=np.asarray(probe['scale'],dtype=np.float32)
PROBE_COEF=np.asarray(probe['coef'],dtype=np.float32);PROBE_INTERCEPT=float(probe['intercept'])
assert PROBE_MEAN.shape==PROBE_SCALE.shape==PROBE_COEF.shape and (PROBE_SCALE>0).all()
assert all(np.isfinite(x).all() for x in (PROBE_MEAN,PROBE_SCALE,PROBE_COEF))
BALANCE_THRESHOLD=float(old_protocol['balance_threshold']);READOUT_TOLERANCE=float(old_protocol['readout_tolerance'])
manifest_path=PROJECT_ROOT/'results/behavior_comparison_v3/activation_pilot_manifest.jsonl'
manifest=read_jsonl(manifest_path)
dev_ids={str(r['scene_id']) for r in manifest if r['split']=='probe_dev'}
assert len(dev_ids)==19 and len(manifest)==254
assert dev_ids<=set(probe['fit_scene_ids'])
cfg=BehaviorRunConfig(dataset_dir=DATASET_DIR,output_dir=OUTPUT_ROOT/'unused',model_id=MODEL_ID,model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,),question_kinds=('outcome',),splits=('probe_dev',),batch_size=1,generation_every=0,seed=20260820)
assert cfg.dataset_fingerprint()==old_runtime['dataset'], 'Dataset differs from the original E1 inputs.'
by_scene=defaultdict(list)
for ex in load_behavior_examples(cfg):
    if str(ex.scene_id) in dev_ids:by_scene[str(ex.scene_id)].append(ex)
assert set(by_scene)==dev_ids and all(len(v)==6 and len({e.variant for e in v})==6 for v in by_scene.values())
for examples in by_scene.values():assert len({tuple(e.option_semantics) for e in examples})==6
print(f'只用旧开发集：19 场景 × 6 映射 × {REPEATS} 次独立前向。冻结读出层：{SELECTED_LAYER}。')


Load the original frozen readout and nineteen development scenes:  50%|#####     | 6/12 [00:00<?, ?cell/s]

只用旧开发集：19 场景 × 6 映射 × 3 次独立前向。冻结读出层：13。


In [9]:
%%cell_progress 8/12 Load the pinned model
cell_stage(f'Loading {MODEL_KEY} weights for depth localization')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)
assert adapter.metadata['resolved_revision'] == MODEL_REVISION

def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers',
        'model.layers', 'language_model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
assert all(0 <= layer < len(DECODER_LAYERS) for layer in CAPTURE_LAYERS)
print('Layer path:', LAYER_PATH)
print('Capture layers:', CAPTURE_LAYERS)
assert not any(v == 'disk' for v in getattr(adapter.model,'hf_device_map',{}).values()), 'Use a GPU with enough memory for this model.'


Load the pinned model:  58%|#####8    | 7/12 [00:00<?, ?cell/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

{'adapter': 'HuggingFaceMultimodalAdapter',
 'model_id': 'Qwen/Qwen3.5-9B',
 'requested_revision': 'c202236235762e1c871ad0ccb60c8ee5ba337b9a',
 'resolved_revision': 'c202236235762e1c871ad0ccb60c8ee5ba337b9a',
 'dtype': 'bfloat16',
 'device_map': 'auto',
 'parameter_count': 9409813744,
 'torch_version': '2.11.0+cu128',
 'transformers_version': '5.14.0'}

Layer path: model.language_model.layers
Capture layers: (13,)


In [10]:
%%cell_progress 9/12 Use the original E1 capture and frozen decoder
def output_hidden(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and torch.is_tensor(output[0]):
        hidden = output[0]
    else:
        raise TypeError(f'Unsupported decoder output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_hidden(output, value):
    if torch.is_tensor(output):
        return value
    if isinstance(output, tuple):
        return (value,) + output[1:]
    if isinstance(output, list):
        return [value] + output[1:]
    raise TypeError(type(output).__name__)

def forward_one(example, layers=(), edit=None):
    plan = adapter._candidate_plan(example)
    assert plan is not None, 'A/B/C must have the existing single-token scoring plan'
    prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(adapter._to_device(adapter._template(
        [example.messages()], add_generation_prompt=True)), prefix)
    assert bool(inputs['attention_mask'][0, -1]), 'Final position is padded'
    captured, audits, handles = {}, {}, []
    for layer in layers:
        def hook(_module, _inputs, output, layer=layer):
            hidden = output_hidden(output)
            assert hidden.dtype == torch.bfloat16, hidden.dtype
            pre = hidden[0, -1].detach().float().cpu().numpy().copy()
            captured[layer] = pre
            if edit is None:
                return None
            assert len(layers) == 1
            cast = lambda x: torch.as_tensor(np.asarray(x).copy(), dtype=hidden.dtype).float().numpy()
            post, audit = edit(pre, cast)
            modified = hidden.clone()
            modified[0, -1] = torch.as_tensor(post, dtype=hidden.dtype, device=hidden.device)
            actual = modified[0, -1].detach().float().cpu().numpy()
            audit.update(actual_delta_norm=float(np.linalg.norm(actual - pre)),
                         standardized_delta_norm=float(np.linalg.norm((actual-pre)/PROBE_SCALE)),
                         decoded_pre=decode(pre), decoded_post=decode(actual),
                         hidden_norm=float(np.linalg.norm(pre)))
            audits.update(audit)
            return replace_hidden(output, modified)
        handles.append(DECODER_LAYERS[layer].register_forward_hook(hook))
    try:
        with torch.inference_mode():
            output = adapter.model(**inputs, use_cache=False)
            scores = torch.log_softmax(output.logits[0, -1].float(), dim=-1)
            label_scores = {label: float(scores[token_by_label[label]].cpu()) for label in example.labels}
    finally:
        for handle in handles:
            handle.remove()
    assert set(captured) == set(layers)
    predicted_label = max(example.labels, key=lambda label: label_scores[label])
    semantic_scores = {str(example.label_to_semantic[label]): value for label,value in label_scores.items()}
    result = dict(example_id=example.example_id, scene_id=str(example.scene_id), variant=example.variant,
        split=example.split, problem_type=example.problem_type, net_torque=int(example.net_torque),
        gold_semantic=str(example.gold_semantic), predicted_semantic=str(example.label_to_semantic[predicted_label]),
        correct=str(example.label_to_semantic[predicted_label]) == str(example.gold_semantic),
        label_to_semantic={str(k):str(v) for k,v in example.label_to_semantic.items()},
        candidate_logprobs=label_scores, semantic_logprobs=semantic_scores, **audits)
    return result, captured

def readout_value(hidden, mean, scale, coef, intercept):
    return float(((np.asarray(hidden, dtype=np.float64) - mean) / scale) @ coef + intercept)

def decode(hidden):
    return readout_value(hidden, PROBE_MEAN, PROBE_SCALE, PROBE_COEF, PROBE_INTERCEPT)

def probe_outcome(value):
    return 'left' if value > BALANCE_THRESHOLD else 'right' if value < -BALANCE_THRESHOLD else 'balanced'

def example_stamp(example):
    return dict(example_id=example.example_id,prompt=example.prompt,image=file_hash(example.image_path),
        labels=list(example.labels),semantics=list(example.option_semantics),gold=str(example.gold_semantic),
        torque=int(example.net_torque),split=example.split)

def current_runtime_identity():
    import transformers
    return dict(torch=torch.__version__,transformers=transformers.__version__,cuda=torch.version.cuda,
        dtype=MODEL_DTYPE,batch_size=1,checkpoint=MODEL_REVISION,layer_path=LAYER_PATH,
        implementation=IMPLEMENTATION_SHA256,
        toolkit={str(p.relative_to(TOOLKIT_ROOT)):file_hash(p) for p in sorted((TOOLKIT_ROOT/'src').rglob('*.py'))})


Use the original E1 capture and frozen decoder:  67%|######6   | 8/12 [00:00<?, ?cell/s]

In [11]:
%%cell_progress 10/12 Lock the current runtime without fitting or retuning the probe
adapter.model.eval()
runtime=current_runtime_identity()
assert runtime['torch']==old_runtime['torch'] and runtime['transformers']==old_runtime['transformers'], '恢复旧 E1 的 torch/transformers 版本后运行此同配置诊断。'
toolkit_identity={str(p.relative_to(TOOLKIT_ROOT)):'sha256:'+file_hash(p) for p in sorted((TOOLKIT_ROOT/'src/visual_torque_toolkit').rglob('*.py'))}
assert toolkit_identity==old_runtime['toolkit']
assert old_runtime['manifest']=='sha256:'+file_hash(manifest_path)
runtime.update(gpu=torch.cuda.get_device_name(0),cuda_capability=list(torch.cuda.get_device_capability(0)),
    attention_implementation=str(getattr(adapter.model.config,'_attn_implementation',None)),
    tf32_matmul=bool(torch.backends.cuda.matmul.allow_tf32),cudnn_benchmark=bool(torch.backends.cudnn.benchmark))
repeat_identity=dict(protocol=PROTOCOL,runtime=runtime,probe=file_hash(probe_path),old_protocol=file_hash(old_protocol_path),
    old_runtime=file_hash(old_runtime_path),manifest=file_hash(manifest_path),
    inputs={s:[example_stamp(e) for e in sorted(by_scene[s],key=lambda e:e.variant)] for s in sorted(by_scene)})
freeze(OUTPUT_ROOT/'repeat_identity.json',repeat_identity)
print('原读出器已冻结；保存 BF16 前向状态的精确 float32 副本，不作 FP16 存储压缩。')


Lock the current runtime without fitting or retuning the probe:  75%|#######5  | 9/12 [00:00<?, ?cell/s]

原读出器已冻结；保存 BF16 前向状态的精确 float32 副本，不作 FP16 存储压缩。


In [12]:
%%cell_progress 11/12 Run three independent forwards per mapping and preserve raw states
repeat_records=[];states_by_scene={}
for sid in tqdm(sorted(by_scene),desc='Same-configuration independent repeats'):
    examples=sorted(by_scene[sid],key=lambda e:e.variant)
    array_path=OUTPUT_ROOT/'states'/(digest(sid)[:24]+'.npz')
    def compute(examples=examples):
        states=[];rows=[]
        for repeat in range(REPEATS):
            current=[]
            for ex in examples:
                record,h=forward_one(ex,(SELECTED_LAYER,))
                hidden=h[SELECTED_LAYER].astype('float32')
                current.append(hidden)
                rows.append(dict(scene_id=str(ex.scene_id),variant=ex.variant,repeat=repeat,
                    decoded=decode(hidden),decoded_outcome=probe_outcome(decode(hidden)),record=record))
            states.append(current)
        states=np.asarray(states,dtype='float32')
        assert states.shape[:2]==(REPEATS,6) and np.isfinite(states).all()
        # States and scores commit together. A crash cannot leave a completed array paired with different rerun scores.
        return dict(records=rows,states=states.tolist())
    cached=cached_json(OUTPUT_ROOT/'repeat_shards',sid,repeat_identity,compute)
    states=np.asarray(cached['states'],dtype='float32')
    frozen_arrays(array_path,{'states':states})
    states_by_scene[sid]=states
    repeat_records.extend(cached['records'])
assert len(repeat_records)==19*6*REPEATS
save_json(ANALYSIS_DIR/'repeat_records.json',repeat_records)


Run three independent forwards per mapping and preserve raw states:  83%|########3 | 10/12 [00:00<?, ?cell/s]

Same-configuration independent repeats:   0%|          | 0/19 [00:00<?, ?it/s]

In [13]:
%%cell_progress 12/12 Separate within-runtime variation from historical scene-mean differences
rows=[];scene_rows=[]
for sid in sorted(by_scene):
    records={(r['repeat'],r['variant']):r for r in repeat_records if r['scene_id']==sid}
    variants=[e.variant for e in sorted(by_scene[sid],key=lambda e:e.variant)]
    states=states_by_scene[sid]
    for a,b in combinations(range(REPEATS),2):
        for i,variant in enumerate(variants):
            left,right=records[a,variant],records[b,variant]
            x=np.array([left['record']['candidate_logprobs'][k] for k in LABELS])
            y=np.array([right['record']['candidate_logprobs'][k] for k in LABELS])
            decoded_delta=abs(left['decoded']-right['decoded'])
            contributions=(states[b,i].astype(float)-states[a,i].astype(float))*PROBE_COEF/PROBE_SCALE
            gap=float(np.max(abs((x-x.mean())-(y-y.mean()))))
            rows.append(dict(scene_id=sid,variant=variant,repeat_a=a,repeat_b=b,readout_absolute_change=decoded_delta,
                readout_over_tolerance=decoded_delta>READOUT_TOLERANCE,
                decoded_outcome_changed=left['decoded_outcome']!=right['decoded_outcome'],
                answer_changed=left['record']['predicted_semantic']!=right['record']['predicted_semantic'],
                centered_score_max_change=gap,hidden_max_change=float(np.max(abs(states[b,i]-states[a,i]))),
                effective_coordinate_l1=float(abs(contributions).sum()),
                largest_readout_contribution=float(abs(contributions).max())))
        x=decode(states[a].mean(0));y=decode(states[b].mean(0))
        scene_rows.append(dict(scene_id=sid,repeat_a=a,repeat_b=b,scene_mean_readout_change=abs(x-y),
            scene_mean_class_changed=probe_outcome(x)!=probe_outcome(y)))
changes=pd.DataFrame(rows);means=pd.DataFrame(scene_rows)
changes.to_csv(ANALYSIS_DIR/'within_runtime_pairs.csv',index=False)
means.to_csv(ANALYSIS_DIR/'within_runtime_scene_means.csv',index=False)
summary=[]
for metric in ('readout_absolute_change','readout_over_tolerance','decoded_outcome_changed','answer_changed','centered_score_max_change','hidden_max_change'):
    maxima=changes.groupby('scene_id')[metric].max()
    summary.append(dict(metric=metric,scenes=len(maxima),mean_of_scene_maxima=float(maxima.mean()),
        maximum=float(maxima.max()),nonzero_scenes=int(maxima.ne(0).sum())))
pd.DataFrame(summary).to_csv(ANALYSIS_DIR/'repeatability_summary.csv',index=False)
archive_comparison=[]
discovery=PROJECT_ROOT/'results'/MODEL_SLUG/'activation_discovery_v1'
archive_path=discovery/'scene_mean_activations.pt';config_path=discovery/'capture_config.json'
archive_status='missing_optional_original_discovery_archive'
if archive_path.is_file() and config_path.is_file():
    identity=read_json(config_path)['identity']
    assert identity['model_id']==MODEL_ID and identity['resolved_revision']==MODEL_REVISION
    assert identity['activation_manifest_sha256']=='sha256:'+file_hash(manifest_path)
    assert identity['dataset_fingerprint']==old_runtime['dataset']
    for path in (archive_path,config_path):
        expected=probe.get('fit_provenance',{}).get('files',{}).get(str(path))
        if expected is not None:assert expected=='sha256:'+file_hash(path), f'Original probe input changed: {path}'
    # This is a user-owned original archive; CPU-load only, no new fitting.
    archived=torch.load(archive_path,map_location='cpu',weights_only=False)
    meta=pd.DataFrame(archived['metadata']);assert meta.scene_id.astype(str).is_unique
    pos=list(map(int,archived['layer_indices'])).index(SELECTED_LAYER)
    old_h=archived['activations'][:,pos].float().numpy()
    for sid in sorted(by_scene):
        indices=np.flatnonzero(meta.scene_id.astype(str).eq(sid).to_numpy());assert len(indices)==1
        ix=int(indices[0]);assert meta.iloc[ix]['split']=='probe_dev'
        old_value=decode(old_h[ix])
        for repeat in range(REPEATS):
            current=states_by_scene[sid][repeat]
            exact_mean=current.mean(0)
            stored_mean=current.astype('float16').astype('float32').mean(0).astype('float16').astype('float32')
            assert np.isfinite(stored_mean).all(), 'FP16 storage diagnostic overflowed; preserve native results and inspect.'
            archive_comparison.append(dict(scene_id=sid,repeat=repeat,archived_readout=old_value,
                current_native_readout=decode(exact_mean),current_fp16_pipeline_readout=decode(stored_mean),
                archived_minus_current=old_value-decode(exact_mean),
                storage_roundtrip_change=decode(stored_mean)-decode(exact_mean)))
    archive_status='compared_historical_scene_means_separate_from_same_runtime_repeats'
    freeze(OUTPUT_ROOT/'archive_inputs.json',dict(archive=file_hash(archive_path),config=file_hash(config_path),identity=identity))
pd.DataFrame(archive_comparison).to_csv(ANALYSIS_DIR/'historical_scene_mean_comparison.csv',index=False)
result=dict(status='complete',model=MODEL_KEY,development_scenes=19,mappings_per_scene=6,repeats=REPEATS,
    forward_count=len(repeat_records),readout_over_tolerance=bool(changes.readout_over_tolerance.any()),
    any_answer_changed=bool(changes.answer_changed.any()),any_readout_class_changed=bool(changes.decoded_outcome_changed.any()),
    any_scene_mean_class_changed=bool(means.scene_mean_class_changed.any()),
    any_score_gap_over_0_15=bool(changes.centered_score_max_change.gt(.15).any()),archive_status=archive_status,
    probe_refit=False,new_scientific_cohort=False)
save_json(ANALYSIS_DIR/'RUN_COMPLETE.json',result)
lines=['# 读出重复性诊断','',json.dumps(result,ensure_ascii=False,indent=2),'',
    '同配置重复变化与历史缓存差异分开报告。没有重复变化不等于已解释旧缓存差异。',
    '本实验只复用旧开发场景，不新增科学测试集，不重新拟合预测器，也不修改旧实验结果。',
    '若读出变化超过原 0.05 容差或改变物理分类，先定位数值来源，再决定重跑哪些受影响条件。',
    '历史 scene-mean 对比保留其原捕获配置；不能仅凭相关差异把原因归给 FP16 存储。']
(ANALYSIS_DIR/'REPORT_ZH.md').write_text('\n\n'.join(lines)+'\n',encoding='utf-8')
display(pd.DataFrame(summary));display(pd.DataFrame(archive_comparison).head());print(result);print(ANALYSIS_DIR)


Separate within-runtime variation from historical scene-mean differences:  92%|#########1| 11/12 [00:00<?, ?ce…

,metric,scenes,mean_of_scene_maxima,maximum,nonzero_scenes
0,readout_absolute_change,19,0.0,0.0,0
1,readout_over_tolerance,19,0.0,0.0,0
2,decoded_outcome_changed,19,0.0,0.0,0
3,answer_changed,19,0.0,0.0,0
4,centered_score_max_change,19,0.0,0.0,0
5,hidden_max_change,19,0.0,0.0,0


,scene_id,repeat,archived_readout,current_native_readout,current_fp16_pipeline_readout,archived_minus_current,storage_roundtrip_change
0,multi_03b698958cd84596,0,-28.014347,-25.382525,-25.365719,-2.631822,0.016807
1,multi_03b698958cd84596,1,-28.014347,-25.382525,-25.365719,-2.631822,0.016807
2,multi_03b698958cd84596,2,-28.014347,-25.382525,-25.365719,-2.631822,0.016807
3,multi_c0e457437617298d,0,-2.020667,-1.794490,-1.793668,-0.226177,0.000822
4,multi_c0e457437617298d,1,-2.020667,-1.794490,-1.793668,-0.226177,0.000822


{'status': 'complete', 'model': 'qwen9b', 'development_scenes': 19, 'mappings_per_scene': 6, 'repeats': 3, 'forward_count': 342, 'readout_over_tolerance': False, 'any_answer_changed': False, 'any_readout_class_changed': False, 'any_scene_mean_class_changed': False, 'any_score_gap_over_0_15': False, 'archive_status': 'compared_historical_scene_means_separate_from_same_runtime_repeats', 'probe_refit': False, 'new_scientific_cohort': False}
/content/drive/MyDrive/VisualTorque/results/cross_model_independent_confirmation/readout_repeatability/main/Qwen__Qwen3.5-9B/analysis
